In [1]:
#=========================================================
# stitch_files.py
# Author: McKenna W. Stanford
# Date Created: 03/17/2023
#=========================================================

In [1]:
#=======================================
# Imports
#=======================================
import numpy as np
import glob
import xarray
import datetime
import pandas as pd
import os
import time
from dask.distributed import Client
import matplotlib.pyplot as plt

In [2]:
#=======================================
# Get list of PLT files
#=======================================
sim_name = 'sip_10x_bin_ice_noturb'

#path = '/pscratch/sd/m/mckenna/dharma_output/'+sim_name+'/restart/'
path = '/pscratch/sd/m/mckenna/dharma_output/'+sim_name+'/'
plt_files = sorted(glob.glob(path+'dharma_PLT_*'))
num_plt_files = len(plt_files)

new_plt_files = []
for ii in range(num_plt_files):
    dum = plt_files[ii].split('/')[-1].split('.')[0].split('_')[-2]
    #if dum < 32400:
    #    continue
    #else:
    new_plt_files.append(plt_files[ii])
    dum = int(dum)
plt_files = new_plt_files
num_plt_files = len(plt_files)
print('# of PLT files:',num_plt_files)

# of PLT files: 21312


In [3]:
#=======================================
# Get list of time and processor number
# values from all files using the file
# name's string.
# Then get unique output times and processor
# numbers
#=======================================
output_times = []
proc_numbers = []
for ii in range(num_plt_files):
    file_str = plt_files[ii]
    # Split it up by '/' to get rid of the path
    file_str = file_str.split('/')[-1]
    # Get rid of ".cdf" extention
    file_str = file_str.split('.')[0]
    # Split up by '_' to get output "time" and processor number
    file_str = file_str.split('_')
    tmp_time = file_str[2]
    tmp_proc = file_str[3]
    output_times.append(tmp_time)
    proc_numbers.append(tmp_proc)
output_times = np.array(output_times)
proc_numbers = np.array(proc_numbers)
unique_output_times = np.unique(output_times)
unique_procs = np.unique(proc_numbers)
num_unique_output_times = len(unique_output_times)
num_unique_procs = len(unique_procs)
print('# of unique output times:',num_unique_output_times)
print('# of processors:',num_unique_procs)

# of unique output times: 37
# of processors: 576


In [4]:
unique_output_times

array(['021600', '022200', '022800', '023400', '024000', '024600',
       '025200', '025800', '026400', '027000', '027600', '028200',
       '028800', '029400', '030000', '030600', '031200', '031800',
       '032400', '033000', '033600', '034200', '034800', '035400',
       '036000', '036600', '037200', '037800', '038400', '039000',
       '039600', '040200', '040800', '041400', '042000', '042600',
       '043200'], dtype='<U6')

In [5]:
#=======================================
# Get Variables from Sounding File
# and add to a dictionary. 
#
#These are used to calculate pressure, theta, and
# temperature in the 3D files and makes
# the heights (zt and zw) as well as
# the reference state density (rhobar)
# accessible.
#=======================================
sounding_file = path + 'dharma.soundings.cdf'
ncfile_sounding = xarray.open_dataset(sounding_file)
zt = ncfile_sounding['zt'].values # m
zw = ncfile_sounding['zw'].values # m
rhobar = ncfile_sounding['rhobar'].values # kg/m^3
sounding_th = ncfile_sounding['th'].values # K
T = ncfile_sounding['T'].values # K
sounding_time = ncfile_sounding['time'].values
theta_00 = ncfile_sounding.attrs['theta_00']
ncfile_sounding.close()


sounding_dict = {'zt':zt,\
                 'zw':zw,\
                 'rhobar':rhobar,\
                 'th':sounding_th,\
                 'T':T,\
                 'time':sounding_time,\
                 'theta_00':theta_00,\
                 }

/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()


In [6]:
def interp_winds(u,v,w,x,xu,y,yv,zt,zw):
    
    from scipy.interpolate import RegularGridInterpolator as rgi
    xi,yi,zti = np.meshgrid(x,y,zt,indexing='ij')

    w_interp_func = rgi((x,y,zw), w)
    w_interp = w_interp_func(np.array([xi,yi,zti]).T).T
    
    u_interp_func = rgi((xu,y,zt), u)
    u_interp = u_interp_func(np.array([xi,yi,zti]).T).T
    
    v_interp_func = rgi((x,yv,zt), v)
    v_interp = v_interp_func(np.array([xi,yi,zti]).T).T
    
    return u_interp,v_interp,w_interp

In [7]:
def calc_pressure_theta(var_dict,sounding_dict,time_id):
    """
    Calculate 3D pressure, potential temperature, and temperature, which requires
    pulling some variables from the sounding file
    """
    # Constants
    Rgas = 8314.3
    Mw = 28.966
    cp = 1004.
    Rd = Rgas/Mw
    grav = 9.81
    p_exner = 1.e5
        
    state_dict = {}
    
    theta_3d = (var_dict['th'] + 1)*sounding_dict['theta_00']
    theta_sound = (sounding_dict['th'] + 1)*sounding_dict['theta_00']
    nz = len(sounding_dict['zt'])
    nx = len(var_dict['th'][:,0,0])
    ny = len(var_dict['th'][0,:,0])     

    
    
    pi_sound = np.zeros(nz)
    for kk in range(nz):
        pi_sound[kk] = sounding_dict['T'][time_id,kk]/theta_sound[time_id,kk]
        
    T_3d = pi_sound*theta_3d 
    
    pi_3d = T_3d/theta_3d
    penv_3d = (pi_3d**(cp/Rd))*p_exner # in Pa

    
    state_dict['pressure'] = penv_3d
    state_dict['temperature'] = T_3d
    state_dict['theta'] = theta_3d
    state_dict['exner'] = pi_3d
    
    return state_dict

In [8]:
def stitch_files(key, kwargs):
    """
    Stitches one variable (key) across all processor files into a single 3D array.

    NOTE: This function is kept for potential future parallel use only.
    In the current serial workflow it is NOT called – stitch_lvl_1 reads
    every processor file once and fills all variables in a single pass,
    which is ~(num_variables)x more I/O-efficient on Lustre.

    If you want to parallelise later, the recommended level is across
    *output times*, not across variables within a single output time:

        futures = client.map(stitch_and_write,
                             unique_output_times,
                             [plt_files]*N, ...)

    Parallelising across variables forces all workers to open the same
    files simultaneously, blowing past the per-process file-handle limit
    and triggering 'NetCDF: Not a valid ID' errors on NERSC Lustre.
    """
    nx = kwargs['nx']
    ny = kwargs['ny']
    nz = kwargs['nz']
    nboxes = kwargs['nboxes']
    chop_files = kwargs['chop_files']

    if   key == 'w': vdata = np.zeros((nx,   ny,   nz+1))
    elif key == 'u': vdata = np.zeros((nx+1, ny,   nz  ))
    elif key == 'v': vdata = np.zeros((nx,   ny+1, nz  ))
    else:            vdata = np.zeros((nx,   ny,   nz  ))

    for jj in range(nboxes):
        ncfile    = xarray.open_dataset(chop_files[jj], engine='scipy')
        tmp_keys  = list(ncfile.keys())
        extension = tmp_keys[0][-5:]
        full_key  = key + extension
        tmp_var   = ncfile[full_key]

        bnds  = tmp_var.attrs['bnds']
        ngrow = tmp_var.attrs['ngrow']
        ncfile.close()

        ib, jb, kb = bnds[0]-1, bnds[1]-1, bnds[2]-1
        ie, je, ke = bnds[3]-1, bnds[4]-1, bnds[5]-1
        mx = ie-ib+1;  my = je-jb+1;  mz = ke-kb+1

        tmp = np.reshape(tmp_var.values, (mx, my, mz), order='F')
        vdata[ib+ngrow:ie-ngrow+1,
              jb+ngrow:je-ngrow+1,
              kb+ngrow:ke-ngrow+1] = tmp[ngrow:mx-ngrow,
                                         ngrow:my-ngrow,
                                         ngrow:mz-ngrow]
    return vdata


In [9]:
def calc_effective_radius(var_dict):
    min_thresh = 1.e-3
    
    tmp_lwc = var_dict['ql']*var_dict['rhobar']*1.e3
    tmp_iwc = var_dict['qif']*var_dict['rhobar']*1.e3
    tmp_rhobar = var_dict['rhobar']

    nz = len(var_dict['zt'])
    nx = len(var_dict['x'])
    ny = len(var_dict['y'])
    
    reff_l = np.zeros((nx,ny,nz))-999.
    reff_l_lt_100 = np.zeros((nx,ny,nz))-999.
    nl_int = np.zeros((nx,ny,nz))-999.
    nl_int_lt_100 = np.zeros((nx,ny,nz))-999.
    
    
    for kk in range(nz):
        lwc_single_level = tmp_lwc[:,:,kk] # g/m3
        rhobar_single_level = tmp_rhobar[kk]
        tmp_var_dict = {}
        for key in var_dict.keys():
            if ('liq_num' in key) & (key != 'tot_liq_num'):
                tmp_var_dict[key] = var_dict[key][:,:,kk]

        diff_rad = np.diff(var_dict['pc_rbound_02']) # um
        rad = var_dict['pc_radius_02'] # um
        num_bins = len(diff_rad) # 50
        
        liq_id = np.where(lwc_single_level > min_thresh)
        if np.size(liq_id) == 0.:
            continue
        #print(kk)

        num_points = np.size(liq_id[0])
        
        dists = np.zeros((num_points,num_bins))
        dumi=0
        for key,val in tmp_var_dict.items():
            dists[:,dumi] = val[liq_id[0],liq_id[1]]*1.e3*rhobar_single_level # liq_num is in #/g so need to convert to #/m3
            dumi+=1
        dists = dists/(diff_rad*1.e-6) # convert from #/m3 to #/m4
        #dists = dists/(rad*1.e-6) # convert from #/m3 to #/m4
        
        dumid = np.squeeze(np.where(rad < 100.))
        dists_lt_100 = dists[:,dumid]
        rad_lt_100 = rad[dumid]
        diff_rad_lt_100 = diff_rad[dumid]

            
        tmp_reff_l = []
        tmp_reff_l_lt_100 = []
        tmp_nl_int = []
        tmp_nl_int_lt_100 = []
        
        for ii in range(len(dists[:,0])):
            dum_reff_l_num = np.trapz(dists[ii,:]*((rad*1.e-6)**3.),rad*1.e-6,dx=diff_rad*1.e-6)
            dum_reff_l_num_lt_100 = np.trapz(dists_lt_100[ii,:]*((rad_lt_100*1.e-6)**3.),rad_lt_100*1.e-6,dx=diff_rad_lt_100*1.e-6)
            dum_reff_l_den = np.trapz(dists[ii,:]*((rad*1.e-6)**2.),rad*1.e-6,dx=diff_rad*1.e-6)
            dum_reff_l_den_lt_100 = np.trapz(dists_lt_100[ii,:]*((rad_lt_100*1.e-6)**2.),rad_lt_100*1.e-6,dx=diff_rad_lt_100*1.e-6)
            
            tmp_nl_int.append(np.trapz(dists[ii,:],rad*1.e-6,dx=diff_rad*1.e-6)) # /m3
            tmp_nl_int_lt_100.append(np.trapz(dists_lt_100[ii,:],rad_lt_100*1.e-6,dx=diff_rad_lt_100*1.e-6)) # /m3

            tmp_reff_l.append((dum_reff_l_num/dum_reff_l_den)*1.e6) # um
            tmp_reff_l_lt_100.append((dum_reff_l_num_lt_100/dum_reff_l_den_lt_100)*1.e6) # um

        tmp_reff_l = np.array(tmp_reff_l)
        tmp_reff_l_lt_100 = np.array(tmp_reff_l_lt_100)
        tmp_nl_int = np.array(tmp_nl_int)*1.e-6 # Convert back to /cc
        tmp_nl_int_lt_100 = np.array(tmp_nl_int_lt_100)*1.e-6 # Convert back to /cc
        
        reff_l[liq_id[0],liq_id[1],kk] = tmp_reff_l
        nl_int[liq_id[0],liq_id[1],kk] = tmp_nl_int
        reff_l_lt_100[liq_id[0],liq_id[1],kk] = tmp_reff_l_lt_100
        nl_int_lt_100[liq_id[0],liq_id[1],kk] = tmp_nl_int_lt_100
        
        
    return reff_l,reff_l_lt_100,nl_int,nl_int_lt_100

In [10]:
def calc_liq_ice_metrics(var_dict, min_thresh=1.e-6):
    """
    Calculate effective radius and median mass diameter for liquid and ice
    from LES bin microphysics output.
    
    All outputs are in SI units:
      - Effective radius (m)
      - Median mass diameter (m)
      - Number concentration (#/m³)
    
    Parameters
    ----------
    var_dict : dict
        Dictionary containing LES microphysics variables.
    min_thresh : float
        Minimum water content (g/m³) to consider a grid point.
        
    Returns
    -------
    reff_l, D50_l, n_l : np.ndarray
        Liquid effective radius [m], median mass diameter [m], number concentration [#/m³]
        Shape: (nx, ny, nz)
        
    reff_i, D50_i, n_i : np.ndarray
        Ice effective radius [m], median mass diameter [m], number concentration [#/m³]
        Shape: (nx, ny, nz)
    """
    
    # Grid dimensions
    nz = len(var_dict['zt'])
    nx = len(var_dict['x'])
    ny = len(var_dict['y'])
    
    # Allocate arrays
    reff_l = np.full((nx,ny,nz), np.nan)
    D50_l = np.full((nx,ny,nz), np.nan)
    n_l = np.full((nx,ny,nz), np.nan)
    
    reff_i = np.full((nx,ny,nz), np.nan)
    D50_i = np.full((nx,ny,nz), np.nan)
    n_i = np.full((nx,ny,nz), np.nan)
    
    # Convert radii to meters
    rad_l = var_dict['pc_radius_02']*1e-6
    rad_i = var_dict['pc_radius_03']*1e-6
    
    # Bin widths (meters)
    dR_l = np.diff(var_dict['pc_rbound_02'])*1e-6
    dR_i = np.diff(var_dict['pc_rbound_03'])*1e-6
    
    # Mass per particle (kg)
    mass_l = var_dict['pc_mass_02']*1e-3
    mass_i = var_dict['pc_mass_03']*1e-3
    
    # Densities (kg/m³)
    rho_l = var_dict['pc_rhop_02']*1000.
    rho_i = var_dict['pc_rhop_03']*1000.
    
    for k in range(nz):
        # Local water content (g/m³)
        lwc = var_dict['ql'][:,:,k]*var_dict['rhobar'][k]*1e3
        iwc = var_dict['qif'][:,:,k]*var_dict['rhobar'][k]*1e3
        rho_air = var_dict['rhobar'][k]  # g/m³

        # Liquid number bins
        num_bins_l = [key for key in var_dict if ('liq_num' in key) and (key != 'tot_liq_num')]
        # Ice number bins
        num_bins_i = [key for key in var_dict if ('ice_num' in key) and (key != 'tot_ice_num')]

        # Liquid
        liq_idx = np.where(lwc > min_thresh)
        if liq_idx[0].size > 0:
            npts = liq_idx[0].size
            # Create array (npts x nbins)
            n_array_l = np.zeros((npts, len(num_bins_l)))
            for i, key in enumerate(num_bins_l):
                # #/g → #/m³
                n_array_l[:,i] = var_dict[key][:,:,k][liq_idx]*1e3*rho_air
                
            # Effective radius: 3rd/2nd moment
            reff_l[liq_idx + (k,)] = np.trapz(n_array_l*(rad_l**3), rad_l, axis=1) / \
                                      np.trapz(n_array_l*(rad_l**2), rad_l, axis=1)
            # Median mass diameter
            cum_mass_l = np.cumsum(n_array_l*mass_l, axis=1)
            total_mass_l = cum_mass_l[:,-1]
            D50_l_tmp = np.zeros(npts)
            for i_pt in range(npts):
                idx = np.searchsorted(cum_mass_l[i_pt,:], 0.5*total_mass_l[i_pt])
                D50_l_tmp[i_pt] = rad_l[idx]*2  # diameter = 2*radius
            D50_l[liq_idx + (k,)] = D50_l_tmp
            n_l[liq_idx + (k,)] = np.trapz(n_array_l, rad_l, axis=1)
        
        # Ice
        ice_idx = np.where(iwc > min_thresh)
        if ice_idx[0].size > 0:
            npts = ice_idx[0].size
            n_array_i = np.zeros((npts, len(num_bins_i)))
            for i, key in enumerate(num_bins_i):
                n_array_i[:,i] = var_dict[key][:,:,k][ice_idx]*1e3*rho_air
                
            reff_i[ice_idx + (k,)] = np.trapz(n_array_i*(rad_i**3), rad_i, axis=1) / \
                                      np.trapz(n_array_i*(rad_i**2), rad_i, axis=1)
            
            # Median mass diameter
            cum_mass_i = np.cumsum(n_array_i*mass_i, axis=1)
            total_mass_i = cum_mass_i[:,-1]
            D50_i_tmp = np.zeros(npts)
            for i_pt in range(npts):
                idx = np.searchsorted(cum_mass_i[i_pt,:], 0.5*total_mass_i[i_pt])
                D50_i_tmp[i_pt] = rad_i[idx]*2
            D50_i[ice_idx + (k,)] = D50_i_tmp
            n_i[ice_idx + (k,)] = np.trapz(n_array_i, rad_i, axis=1)
            
    return reff_l, D50_l, n_l, reff_i, D50_i, n_i


In [11]:
def calc_liq_ice_metrics_and_reflectivity(var_dict, min_lwc_thresh=1.e-6, min_iwc_thresh=1.e-6):
    """
    Calculate effective radius, median mass diameter, number concentration,
    and Rayleigh reflectivity (Z, dBZ) for liquid and ice from LES bin microphysics output.

    Parameters
    ----------
    var_dict : dict
        Dictionary containing LES microphysics variables.
    min_lwc_thresh : float
        Minimum LWC (g/m³) to consider a grid point for liquid calculations.
    min_iwc_thresh : float
        Minimum IWC (g/m³) to consider a grid point for ice calculations.

    Returns
    -------
    reff_l, D50_l, n_l, reff_i, D50_i, n_i,
    Z_l, Z_i, Z_tot, dBZ_l, dBZ_i, dBZ_tot

    Notes on n_l / n_i (ni_int / nl_int)
    --------------------------------------
    The per-bin values ice_num_k / liq_num_k are in #/g_air (number mixing ratio).
    After multiplying by rho_air [g/m³] they become #/m³ — the number already
    integrated over that bin.  To recover the size distribution dN/dR [#/m³/m]
    we divide by the bin width ΔR before integrating over radius:

        n_total = ∫ (N_k / ΔR_k) dR  ≈  Σ N_k   [#/m³]

    This is then converted to #/cm³ (*1e-6).  The reff and Z calculations are
    unchanged: they use N_k directly and the ΔR factors cancel in the ratios.
    """

    # Grid dimensions
    nx = len(var_dict['x'])
    ny = len(var_dict['y'])
    nz = len(var_dict['zt'])

    # Allocate arrays
    reff_l = np.full((nx, ny, nz), np.nan)
    D50_l  = np.full((nx, ny, nz), np.nan)
    n_l    = np.full((nx, ny, nz), np.nan)

    reff_i = np.full((nx, ny, nz), np.nan)
    D50_i  = np.full((nx, ny, nz), np.nan)
    n_i    = np.full((nx, ny, nz), np.nan)

    Z_l = np.zeros((nx, ny, nz), dtype=float)
    Z_i = np.zeros((nx, ny, nz), dtype=float)

    # Bin properties
    rad_l  = var_dict['pc_radius_02'] * 1e-6   # bin-centre radii, µm → m
    rad_i  = var_dict['pc_radius_03'] * 1e-6
    mass_l = var_dict['pc_mass_02']   * 1e-3   # bin mass, g → kg
    mass_i = var_dict['pc_mass_03']   * 1e-3

    # Bin widths (m) — needed to form dN/dR from per-bin totals
    dR_l = np.diff(var_dict['pc_rbound_02']) * 1e-6   # µm → m
    dR_i = np.diff(var_dict['pc_rbound_03']) * 1e-6

    # Diameters in mm for Z calc
    D_mm_l = 2.0 * np.asarray(var_dict['pc_radius_02']) * 1e-3  # mm
    D_mm_i = 2.0 * np.asarray(var_dict['pc_radius_03']) * 1e-3

    # Dielectric factors (Smith 1984)
    K2_l = 0.93
    K2_i = 0.176

    # Bin keys
    num_bins_l = [k for k in var_dict if ('liq_num' in k) and (k != 'tot_liq_num')]
    num_bins_i = [k for k in var_dict if ('ice_num' in k) and (k != 'tot_ice_num')]

    for k in range(nz):
        rho_air_kgm3  = var_dict['rhobar'][k]
        rho_air_gperm3 = rho_air_kgm3 * 1e3

        # Local water contents (g/m³)
        lwc = var_dict['ql'][:, :, k]  * rho_air_kgm3 * 1e3
        iwc = var_dict['qif'][:, :, k] * rho_air_kgm3 * 1e3

        # --- LIQUID ---
        liq_idx = np.where(lwc > min_lwc_thresh)
        if liq_idx[0].size:
            npts = liq_idx[0].size
            # n_array_l[:,i] = N_k [#/m³]: per-bin number concentration
            n_array_l = np.zeros((npts, len(num_bins_l)))
            for i, key in enumerate(num_bins_l):
                n_array_l[:, i] = var_dict[key][:, :, k][liq_idx] * rho_air_gperm3

            # Effective radius (3rd / 2nd moment) — ΔR cancels in ratio, use N_k directly
            reff_l[liq_idx + (k,)] = (
                np.trapz(n_array_l * (rad_l**3), rad_l, axis=1)
                / np.trapz(n_array_l * (rad_l**2), rad_l, axis=1)
            )

            # Median mass diameter
            cum_mass_l   = np.cumsum(n_array_l * mass_l, axis=1)
            total_mass_l = cum_mass_l[:, -1]
            D50_tmp = np.zeros(npts)
            for ipt in range(npts):
                idx = np.searchsorted(cum_mass_l[ipt, :], 0.5 * total_mass_l[ipt])
                D50_tmp[ipt] = rad_l[idx] * 2
            D50_l[liq_idx + (k,)] = D50_tmp

            # Number concentration: form dN/dR [#/m³/m], integrate → #/m³, convert → #/cm³
            n_psd_l = n_array_l / dR_l[None, :]
            n_l[liq_idx + (k,)] = np.trapz(n_psd_l, rad_l, axis=1) * 1e-6

            # Rayleigh reflectivity (mm^6/m^3)
            Z_l[liq_idx[0], liq_idx[1], k] = (
                K2_l * np.sum(n_array_l * (D_mm_l**6)[None, :], axis=1)
            )

        # --- ICE ---
        ice_idx = np.where(iwc > min_iwc_thresh)
        if ice_idx[0].size:
            npts = ice_idx[0].size
            n_array_i = np.zeros((npts, len(num_bins_i)))
            for i, key in enumerate(num_bins_i):
                n_array_i[:, i] = var_dict[key][:, :, k][ice_idx] * rho_air_gperm3

            reff_i[ice_idx + (k,)] = (
                np.trapz(n_array_i * (rad_i**3), rad_i, axis=1)
                / np.trapz(n_array_i * (rad_i**2), rad_i, axis=1)
            )

            cum_mass_i   = np.cumsum(n_array_i * mass_i, axis=1)
            total_mass_i = cum_mass_i[:, -1]
            D50_tmp = np.zeros(npts)
            for ipt in range(npts):
                idx = np.searchsorted(cum_mass_i[ipt, :], 0.5 * total_mass_i[ipt])
                D50_tmp[ipt] = rad_i[idx] * 2
            D50_i[ice_idx + (k,)] = D50_tmp

            # Number concentration: form dN/dR [#/m³/m], integrate → #/m³, convert → #/cm³
            n_psd_i = n_array_i / dR_i[None, :]
            n_i[ice_idx + (k,)] = np.trapz(n_psd_i, rad_i, axis=1) * 1e-6

            Z_i[ice_idx[0], ice_idx[1], k] = (
                K2_i * np.sum(n_array_i * (D_mm_i**6)[None, :], axis=1)
            )

    # Total reflectivity and dBZ conversion
    Z_tot = Z_l + Z_i
    with np.errstate(divide="ignore", invalid="ignore"):
        dBZ_l   = np.where(Z_l   > 0, 10 * np.log10(Z_l),   np.nan)
        dBZ_i   = np.where(Z_i   > 0, 10 * np.log10(Z_i),   np.nan)
        dBZ_tot = np.where(Z_tot > 0, 10 * np.log10(Z_tot),  np.nan)

    return (
        reff_l, D50_l, n_l,
        reff_i, D50_i, n_i,
        Z_l, Z_i, Z_tot,
        dBZ_l, dBZ_i, dBZ_tot
    )


In [12]:
def stitch_lvl_1(in_output_time, plt_file_list, output_time_list, sounding_dict,
                 sim_name, interp_winds_flag):
    """
    Stitch all processor files for one output time into a single var_dict.

    Key design change from the original:
      Each processor file is opened EXACTLY ONCE and ALL variables are
      extracted in that single pass.  The old approach called stitch_files()
      once per variable key, meaning every file was opened ~260 times per
      output time (~144,000 open/close ops total on Lustre).  This version
      does 576 opens total – roughly 250x less file I/O.

    To parallelise later (once serial is verified):
      Wrap stitch_lvl_1 + write_to_file in a helper and map it across
      unique_output_times with ProcessPoolExecutor or dask.distributed.
      Do NOT parallelise across variables within a single output time.
    """

    print('Stitching output time {} for sim: {}'.format(in_output_time, sim_name))
    t_start = time.time()

    # --- Find the chop files for this output time ---
    file_ids  = np.where(output_times == in_output_time)   # uses global output_times
    chop_files = np.array(plt_file_list)[file_ids]
    nboxes     = len(chop_files)

    # --- Read metadata from first file only ---
    ncfile0   = xarray.open_dataset(chop_files[0], engine='scipy')
    xtime = ncfile0.attrs['time']
    nx    = ncfile0.attrs['nx']
    ny    = ncfile0.attrs['ny']
    nz    = ncfile0.attrs['nz']
    L_x   = ncfile0.attrs['L_x']
    L_y   = ncfile0.attrs['L_y']

    # Collect all pc_* global attributes in one shot
    pc_attrs = {k: v for k, v in ncfile0.attrs.items() if k.startswith('pc_')}

    # Derive base variable names by stripping the 5-char processor extension
    # e.g.  'qv_0000' -> 'qv',  'pc_n01_0000' -> 'pc_n01'
    file_var_keys = list(ncfile0.keys())
    all_base_keys = list(dict.fromkeys(k[:-5] for k in file_var_keys))
    ncfile0.close()

    # --- Grid coordinates ---
    dx = L_x / nx;  dy = L_y / ny
    x  = dx/2. + np.arange(nx) * dx
    y  = dy/2. + np.arange(ny) * dy
    xu = np.arange(nx+1) * dx
    yv = np.arange(ny+1) * dy

    # --- Pre-allocate one array per base variable ---
    vdata = {}
    for key in all_base_keys:
        if   key == 'w': vdata[key] = np.zeros((nx,   ny,   nz+1))
        elif key == 'u': vdata[key] = np.zeros((nx+1, ny,   nz  ))
        elif key == 'v': vdata[key] = np.zeros((nx,   ny+1, nz  ))
        else:            vdata[key] = np.zeros((nx,   ny,   nz  ))

    # -------------------------------------------------------
    # Main I/O loop: open each processor file ONCE,
    # fill ALL variables from it, then close.
    # -------------------------------------------------------
    for jj, fpath in enumerate(chop_files):
        ncfile = xarray.open_dataset(fpath, engine='scipy')
        ext    = list(ncfile.keys())[0][-5:]   # e.g. '_0042'

        for key in all_base_keys:
            full_key = key + ext
            if full_key not in ncfile:
                continue
            tmp_var = ncfile[full_key]
            bnds    = tmp_var.attrs['bnds']
            ngrow   = tmp_var.attrs['ngrow']

            ib, jb, kb = bnds[0]-1, bnds[1]-1, bnds[2]-1
            ie, je, ke = bnds[3]-1, bnds[4]-1, bnds[5]-1
            mx = ie-ib+1;  my = je-jb+1;  mz = ke-kb+1

            tmp = np.reshape(tmp_var.values, (mx, my, mz), order='F')
            vdata[key][ib+ngrow:ie-ngrow+1,
                       jb+ngrow:je-ngrow+1,
                       kb+ngrow:ke-ngrow+1] = tmp[ngrow:mx-ngrow,
                                                   ngrow:my-ngrow,
                                                   ngrow:mz-ngrow]
        ncfile.close()

        if (jj+1) % 100 == 0 or jj == nboxes-1:
            print('  {:4d}/{} processor files read  ({:.1f}s)'.format(
                  jj+1, nboxes, time.time()-t_start))

    # --- Separate pc and non-pc base keys ---
    pc_base_keys     = [k for k in all_base_keys if 'pc' in k]
    non_pc_base_keys = [k for k in all_base_keys if 'pc' not in k]

    # --- Build var_dict with non-pc fields ---
    var_dict = {key: vdata[key] for key in non_pc_base_keys}

    # --- Classify pc variables by index (50 bins each, same order as DHARMA output) ---
    #     0:50   -> aero_num
    #     50:100 -> liq_num
    #     100:150-> liq_core_mass
    #     150:200-> ice_num
    #     200:250-> ice_core_mass
    aero_num_keys      = []
    liq_num_keys       = []
    liq_core_mass_keys = []
    ice_num_keys       = []
    ice_core_mass_keys = []
    dumi = dumii = dumiii = dumiv = dumv = 1

    for ii, key in enumerate(pc_base_keys):
        if ii < 50:
            k = 'aero_num_'     + str(dumi);    aero_num_keys.append(k);       var_dict[k] = vdata[key]; dumi   += 1
        elif ii < 100:
            k = 'liq_num_'      + str(dumii);   liq_num_keys.append(k);        var_dict[k] = vdata[key]; dumii  += 1
        elif ii < 150:
            k = 'liq_core_mass_'+ str(dumiii);  liq_core_mass_keys.append(k);  var_dict[k] = vdata[key]; dumiii += 1
        elif ii < 200:
            k = 'ice_num_'      + str(dumiv);   ice_num_keys.append(k);        var_dict[k] = vdata[key]; dumiv  += 1
        else:
            k = 'ice_core_mass_'+ str(dumv);    ice_core_mass_keys.append(k);  var_dict[k] = vdata[key]; dumv   += 1
    del vdata

    # --- Compute totals ---
    tot_aero_num      = sum(var_dict[k] for k in aero_num_keys)
    tot_liq_num       = sum(var_dict[k] for k in liq_num_keys)
    tot_liq_core_mass = sum(var_dict[k] for k in liq_core_mass_keys)
    tot_ice_num       = sum(var_dict[k] for k in ice_num_keys)
    tot_ice_core_mass = sum(var_dict[k] for k in ice_core_mass_keys)

    # Convert #/g -> #/cc  (uses global rhobar[0,:] as in original code)
    tot_liq_num  *= 1.e3;  tot_ice_num  *= 1.e3;  tot_aero_num *= 1.e3
    for kk in range(nz):
        tot_aero_num[:,:,kk] *= rhobar[0,kk] * 1.e-6
        tot_liq_num[ :,:,kk] *= rhobar[0,kk] * 1.e-6
        tot_ice_num[ :,:,kk] *= rhobar[0,kk] * 1.e-6

    var_dict['tot_aero_num']      = tot_aero_num
    var_dict['tot_liq_num']       = tot_liq_num
    var_dict['tot_ice_num']       = tot_ice_num
    var_dict['tot_liq_core_mass'] = tot_liq_core_mass
    var_dict['tot_ice_core_mass'] = tot_ice_core_mass

    # --- Pressure / theta / temperature ---
    time_id = np.where(sounding_dict['time'] == xtime)[0][0]
    state_dict = calc_pressure_theta(var_dict, sounding_dict, time_id)
    var_dict.update(state_dict)

    # --- Heights and reference density ---
    var_dict['zt']     = sounding_dict['zt']
    var_dict['zw']     = sounding_dict['zw']
    var_dict['rhobar'] = sounding_dict['rhobar'][time_id, :]

    # --- Optional wind interpolation ---
    if interp_winds_flag:
        u_i, v_i, w_i = interp_winds(var_dict['u'], var_dict['v'], var_dict['w'],
                                      x, xu, y, yv, var_dict['zt'], var_dict['zw'])
        var_dict['u_interp'] = u_i
        var_dict['v_interp'] = v_i
        var_dict['w_interp'] = w_i

    var_dict['time'] = xtime

    # --- pc attributes (radii, masses, densities, fall speeds, etc.) ---
    var_dict.update(pc_attrs)

    # --- Grid coordinates ---
    var_dict['x']  = x;   var_dict['y']  = y
    var_dict['xu'] = xu;  var_dict['yv'] = yv
    var_dict['nx'] = nx

    # --- Derived microphysics fields ---
    reff_l, mmd_l, nl_int, reff_i, mmd_i, ni_int, \
        Z_l, Z_i, Z_tot, dBZ_l, dBZ_i, dBZ_tot = \
        calc_liq_ice_metrics_and_reflectivity(var_dict)

    var_dict['reff_liq'] = reff_l * 1.e6
    var_dict['mmd_liq']  = mmd_l  * 1.e6
    var_dict['nl_int']   = nl_int
    var_dict['reff_ice'] = reff_i * 1.e6
    var_dict['mmd_ice']  = mmd_i  * 1.e6
    var_dict['ni_int']   = ni_int
    var_dict['dBZ_liq']  = dBZ_l
    var_dict['dBZ_ice']  = dBZ_i
    var_dict['dBZ_tot']  = dBZ_tot

    print('  Done. Total wall time: {:.1f}s'.format(time.time()-t_start))
    return var_dict


In [13]:
def write_to_file(var_dict,sim_name,in_output_time):
    
    
    coords=dict(x=(["nx"],var_dict['x'],{"units":"meters","long_name":"Grid point centers x direction for state variable grid"}),\
        xu=(["nxu"],var_dict['xu'],{"units":"meters","long_name":"Grid point centers for staggered zonal velocity grid"}),\
        y=(["ny"],var_dict['y'],{"units":"meters","long_name":"Grid point centers for state variable grid"}),\
        yv=(["nyv"],var_dict['yv'],{"units":"meters","long_name":"Grid point centers for staggered meridional velocitygrid"}),\
        zt=(["nzt"],var_dict['zt'],{"units":"meters","long_name":"Height for state variable grid"}),\
        zw=(["nzw"],var_dict['zw'],{"units":"meters","long_name":"Height for staggered vertical velocity grid"}),\
        time=(["nt"],[var_dict['time']],{"units":"Seconds since start of simulation","long_name":"Time"}),\
        pc_radius_01=(["pc_radius_01"],var_dict['pc_radius_01'],{"units":"um","long_name":"Aerosol radius at center of size bins"}),\
        pc_rbound_01=(["pc_rbound_01"],var_dict['pc_rbound_01'],{"units":"um","long_name":"Aerosol radius at edges of size bins"}),\
        pc_radius_02=(["pc_radius_02"],var_dict['pc_radius_02'],{"units":"um","long_name":"Liquid radius at center of size bins"}),\
        pc_rbound_02=(["pc_rbound_02"],var_dict['pc_rbound_02'],{"units":"um","long_name":"Liquid radius at edges of size bins"}),\
        pc_radius_03=(["pc_radius_03"],var_dict['pc_radius_03'],{"units":"um","long_name":"Ice radius at center of size bins"}),\
        pc_rbound_03=(["pc_rbound_03"],var_dict['pc_rbound_03'],{"units":"um","long_name":"Ice radius at edges of size bins"}),\
        pc_mass_01=(["pc_mass_01"],var_dict['pc_mass_01'],{"units":"g","long_name":"Aerosol mass bins"}),\
        pc_mass_02=(["pc_mass_02"],var_dict['pc_mass_02'],{"units":"g","long_name":"Liquid mass bins"}),\
        pc_mass_03=(["pc_mass_03"],var_dict['pc_mass_03'],{"units":"g","long_name":"Ice mass bins"}),\
        pc_vfall_01=(["pc_vfall_01"],var_dict['pc_vfall_01'],{"units":"m/s","long_name":"Aerosol terminal fall velocity"}),\
        pc_vfall_02=(["pc_vfall_02"],var_dict['pc_vfall_02'],{"units":"m/s","long_name":"Liquid terminal fall velocity"}),\
        pc_vfall_03=(["pc_vfall_03"],var_dict['pc_vfall_03'],{"units":"m/s","long_name":"Ice terminal fall velocity"}),\
        pc_area_01=(["pc_area_01"],var_dict['pc_area_01'],{"units":"m2","long_name":"Aerosol area at center of size bins"}),\
        pc_area_02=(["pc_area_02"],var_dict['pc_area_02'],{"units":"m2","long_name":"Liquid area at center of size bins"}),\
        pc_area_03=(["pc_area_03"],var_dict['pc_area_03'],{"units":"m2","long_name":"Ice area at center of size bins"}),\
        pc_areab_01=(["pc_areab_01"],var_dict['pc_areab_01'],{"units":"m2","long_name":"Aerosol area at edges of size bins"}),\
        pc_areab_02=(["pc_areab_02"],var_dict['pc_areab_02'],{"units":"m2","long_name":"Liquid area at edges of size bins"}),\
        pc_areab_03=(["pc_areab_03"],var_dict['pc_areab_03'],{"units":"m2","long_name":"Ice area at edges of size bins"}),\
        pc_aspect_01=(["pc_aspect_01"],var_dict['pc_aspect_01'],{"units":"-","long_name":"Aerosol aspect ratio at center of size bins"}),\
        pc_aspect_02=(["pc_aspect_02"],var_dict['pc_aspect_02'],{"units":"-","long_name":"Liquid aspect ratio at center of size bins"}),\
        pc_aspect_03=(["pc_aspect_03"],var_dict['pc_aspect_03'],{"units":"-","long_name":"Ice aspect ratio at center of size bins"}),\
        pc_aspectb_01=(["pc_aspectb_01"],var_dict['pc_aspectb_01'],{"units":"-","long_name":"Aerosol aspect ratio at edges of size bins"}),\
        pc_aspectb_02=(["pc_aspectb_02"],var_dict['pc_aspectb_02'],{"units":"-","long_name":"Liquid aspect ratio at edges of size bins"}),\
        pc_aspectb_03=(["pc_aspectb_03"],var_dict['pc_aspectb_03'],{"units":"-","long_name":"Ice aspect ratio at edges of size bins"}),\
        pc_rhop_01=(["pc_rhop_01"],var_dict['pc_rhop_01'],{"units":"kg/m3","long_name":"Aerosol bulk density"}),\
        pc_rhop_02=(["pc_rhop_02"],var_dict['pc_rhop_02'],{"units":"kg/m3","long_name":"Liquid bulk density"}),\
        pc_rhop_03=(["pc_rhop_03"],var_dict['pc_rhop_03'],{"units":"kg/m3","long_name":"Ice bulk density"}),\
    )
    
    data_vars = dict(
            pressure=(["nx","ny","nzt"],var_dict["pressure"],{"units":"Pa","long_name":"Atmospheric pressure"}),
            theta=(["nx","ny","nzt"],var_dict["theta"],{"units":"K","long_name":"Atmospheric potential temperature"}),
            temperature=(["nx","ny","nzt"],var_dict["temperature"],{"units":"K","long_name":"Atmospheric temperature"}),
            u=(["nxu","ny","nzt"],var_dict["u"],{"units":"m/s","long_name":"Zonal velocity on native staggered grid"}),
            u_interp=(["nx","ny","nzt"],var_dict["u_interp"],{"units":"m/s","long_name":"Zonal velocity interpolated on regular grid"}),
            v=(["nx","nyv","nzt"],var_dict["v"],{"units":"m/s","long_name":"Meridional velocity on native staggered grid"}),
            v_interp=(["nx","ny","nzt"],var_dict["v_interp"],{"units":"m/s","long_name":"Meridional velocity interpolated on regular grid"}),
            w=(["nx","ny","nzw"],var_dict["w"],{"units":"m/s","long_name":"Vertical velocity on native staggered grid"}),
            w_interp=(["nx","ny","nzt"],var_dict["w_interp"],{"units":"m/s","long_name":"Vertical velocity on interpolated on regular grid"}),
            qv=(["nx","ny","nzt"],var_dict["qv"],{"units":"kg/kg","long_name":"Water vapor mixing ratio"}),
            ql=(["nx","ny","nzt"],var_dict["ql"],{"units":"kg/kg","long_name":"Liquid mass mixing ratio"}),
            qi=(["nx","ny","nzt"],var_dict["qif"],{"units":"kg/kg","long_name":"Ice mass mixing ratio"}),
            nl=(["nx","ny","nzt"],var_dict["tot_liq_num"],{"units":"#/cm3","long_name":"Liquid number concentration"}),
            ni=(["nx","ny","nzt"],var_dict["tot_ice_num"],{"units":"#/cm3","long_name":"Ice number concentration"}),
            na=(["nx","ny","nzt"],var_dict["tot_aero_num"],{"units":"#/cm3","long_name":"Aerosol number concentration"}),
            liq_core_mass=(["nx","ny","nzt"],var_dict["tot_liq_core_mass"],{"units":"kg/kg","long_name":"Liquid core mass"}),
            ice_core_mass=(["nx","ny","nzt"],var_dict["tot_ice_core_mass"],{"units":"kg/kg","long_name":"Ice core mass"}),
            nl_int=(["nx","ny","nzt"],var_dict["nl_int"],{"units":"#/cm3","long_name":"Liquid number concentration calculated via integration"}),
            ni_int=(["nx","ny","nzt"],var_dict["ni_int"],{"units":"#/cm3","long_name":"Ice number concentration calculated via integration"}),
            reff_liq=(["nx","ny","nzt"],var_dict["reff_liq"],{"units":"um","long_name":"Liquid effective radius calculated via integration"}),
            reff_ice=(["nx","ny","nzt"],var_dict["reff_ice"],{"units":"um","long_name":"Ice effective radius calculated via integration"}),
            mmd_liq=(["nx","ny","nzt"],var_dict["mmd_liq"],{"units":"um","long_name":"Liquid median mass diameter"}),
            mmd_ice=(["nx","ny","nzt"],var_dict["mmd_ice"],{"units":"um","long_name":"Ice median mass diameter"}),
            dBZ_liq=(["nx","ny","nzt"],var_dict["dBZ_liq"],{"units":"dBZ","long_name":"Liquid-only Rayleigh reflectivity"}),
            dBZ_ice=(["nx","ny","nzt"],var_dict["dBZ_ice"],{"units":"dBZ","long_name":"Ice-only Rayleigh reflectivity"}),
            dBZ_tot=(["nx","ny","nzt"],var_dict["dBZ_tot"],{"units":"dBZ","long_name":"Total Rayleigh reflectivity"}),
            rhobar=(["nzt"],var_dict["rhobar"],{"units":"kg/m^3","long_name":"Reference air density"}),
        
    )

    for key in var_dict.keys():
        if ('aero_num' in key) and (key != 'tot_aero_num'):
            bin_num = int(key.split('_')[-1])
            bin_num = str(bin_num).zfill(2)
            tmp_key = 'aero_num_{}'.format(bin_num)
            data_vars[tmp_key] = (["nx","ny","nzt"],var_dict[key],{"units":"#/g","long_name":"Aerosol number concentration in bin {}".format(bin_num)})
        if ('liq_num' in key) and (key != 'tot_liq_num'):
            bin_num = int(key.split('_')[-1])
            bin_num = str(bin_num).zfill(2)
            tmp_key = 'liq_num_{}'.format(bin_num)
            data_vars[tmp_key] = (["nx","ny","nzt"],var_dict[key],{"units":"#/g","long_name":"Liquid number concentration in bin {}".format(bin_num)})
        if ('liq_core_mass' in key) and (key != 'tot_liq_core_mass'):
            bin_num = int(key.split('_')[-1])
            bin_num = str(bin_num).zfill(2)
            tmp_key = 'liq_core_mass_{}'.format(bin_num)
            data_vars[tmp_key] = (["nx","ny","nzt"],var_dict[key],{"units":"kg/kg","long_name":"Liquid core mass in bin {}".format(bin_num)})        
        if ('ice_num' in key) and (key != 'tot_ice_num'):
            bin_num = int(key.split('_')[-1])
            bin_num = str(bin_num).zfill(2)
            tmp_key = 'ice_num_{}'.format(bin_num)
            data_vars[tmp_key] = (["nx","ny","nzt"],var_dict[key],{"units":"#/g","long_name":"Ice number concentration in bin {}".format(bin_num)})
        if ('ice_core_mass' in key) and (key != 'tot_ice_core_mass'):
            bin_num = int(key.split('_')[-1])
            bin_num = str(bin_num).zfill(2)
            tmp_key = 'ice_core_mass_{}'.format(bin_num)
            data_vars[tmp_key] = (["nx","ny","nzt"],var_dict[key],{"units":"kg/kg","long_name":"Ice core mass in bin {}".format(bin_num)})        
                
        
    current_time = datetime.datetime.now()
    current_time = current_time.strftime('%m/%d/%Y %H:%M EST')
    attrs={'creation_date':current_time, 
             'author':'McKenna W. Stanford', 
             'email':'mws2175@columbia.edu'}

    ds = xarray.Dataset(data_vars,coords,attrs)
    save_path = '/pscratch/sd/m/mckenna/dharma_3d/'+sim_name+'/'
    #dum_sim_name = 'bin_'+sim_name
    #nc_file_name = 'dharma_3d_{}_{}.nc'.format(dum_sim_name,in_output_time)
    nc_file_name = 'dharma_3d_{}_{}.nc'.format(sim_name,in_output_time)
    ds.to_netcdf(save_path+nc_file_name, mode='w')   


## Run the code here

In [14]:
#==========================
# Initiate Client
#==========================
if False:
    client = Client(n_workers=50)
    client

In [15]:
if False:
    client

In [17]:
import traceback

#======================================================
# Send an output time to the "stitch" function, along with the *total*
# list of PLT files and the *total* list of output times. 
# The function will find the correct files to stitch
# together for a given output time. Note that "output time" means the
# output time in the string, which is the desired output time (e.g., hourly, or
# 3600 seconds) divided by the time step. So, if your output frequency is 1 hour
# and your time step is 2 seconds, the output time would be 3600/2 = 1800.
#======================================================
# Define your simulation name, too.
#unique_output_times = unique_output_times[6:]
#for dum_ii in range(7,len(unique_output_times)):
for dum_ii in range(len(unique_output_times)):
    var_dict = stitch_lvl_1(unique_output_times[dum_ii],plt_files,output_times,sounding_dict,sim_name=sim_name,interp_winds_flag=True)    
    try:
        write_to_file(var_dict,sim_name,unique_output_times[dum_ii])
    except Exception as e:
        print(f"\n*** write_to_file FAILED for output time {unique_output_times[dum_ii]} ***")
        traceback.print_exc()
        raise


Stitching output time 021600 for sim: sip_10x_bin_ice_noturb
   100/576 processor files read  (14.8s)
   200/576 processor files read  (28.0s)
   300/576 processor files read  (41.2s)
   400/576 processor files read  (55.5s)
   500/576 processor files read  (69.2s)
   576/576 processor files read  (79.1s)
  Done. Total wall time: 88.9s
Stitching output time 022200 for sim: sip_10x_bin_ice_noturb
   100/576 processor files read  (14.1s)
   200/576 processor files read  (27.5s)
   300/576 processor files read  (40.5s)
   400/576 processor files read  (53.5s)
   500/576 processor files read  (66.3s)
   576/576 processor files read  (76.0s)
  Done. Total wall time: 86.4s
Stitching output time 022800 for sim: sip_10x_bin_ice_noturb
   100/576 processor files read  (14.5s)
   200/576 processor files read  (27.7s)
   300/576 processor files read  (41.3s)
   400/576 processor files read  (54.2s)
   500/576 processor files read  (67.3s)
   576/576 processor files read  (77.0s)
  Done. Total wal

In [18]:
print('done')

done


In [ ]:
import scipy
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import ndimage
from matplotlib.lines import Line2D

def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx],idx

In [ ]:
iwc = var_dict['qif']*var_dict['rhobar']*1.e3
lwc = var_dict['ql']*var_dict['rhobar']*1.e3
reff_l = var_dict['reff_liq']
reff_i = var_dict['reff_ice']
mmd_l = var_dict['mmd_liq']
mmd_i = var_dict['mmd_ice']
z = var_dict['zt']
zw = var_dict['zw']
iwp = scipy.integrate.trapezoid(iwc,z,axis=2)
lwp = scipy.integrate.trapezoid(lwc,z,axis=2)
x = var_dict['x']
y = var_dict['y']
w = var_dict['w']
max_w = np.max(w,axis=2)
ni = var_dict['tot_ice_num']
nl = var_dict['tot_liq_num']
max_ni = np.max(ni,axis=2)
max_nl = np.max(nl,axis=2)
temp = var_dict['temperature']
dBZ_tot = var_dict['dBZ_tot']


# --- compute domain means/stds (option A & B) ---
mean_iwp = np.nanmean(iwp)
mean_lwp = np.nanmean(lwp)
mean_w   = np.nanmean(max_w)

std_iwp = np.nanstd(iwp)
std_lwp = np.nanstd(lwp)
std_w   = np.nanstd(max_w)

# avoid division by zero
std_iwp = std_iwp if std_iwp > 0 else 1.0
std_lwp = std_lwp if std_lwp > 0 else 1.0
std_w   = std_w   if std_w   > 0 else 1.0

z_iwp = (iwp - mean_iwp) / std_iwp
z_lwp = (lwp - mean_lwp) / std_lwp
z_w   = (max_w - mean_w)   / std_w


# --- Option A: triple standardized product ---
metric_A = z_iwp * z_lwp * z_w
# choose whether to maximize signed value or magnitude:
metric_A_abs = np.abs(metric_A)   # if you want magnitude regardless of sign

# --- Option B: sum of pairwise standardized products ---
metric_B = (z_iwp * z_lwp) + (z_iwp * z_w) + (z_lwp * z_w)

# --- Option C: local (neighborhood) covariances ---
# choose window size (odd integer), e.g. 5x5 neighbourhood
window = 5
footprint = np.ones((window, window)) / (window * window)

# local means
local_mean_iwp = ndimage.convolve(iwp, footprint, mode='mirror')
local_mean_lwp = ndimage.convolve(lwp, footprint, mode='mirror')
local_mean_w   = ndimage.convolve(max_w, footprint, mode='mirror')

# local means of products
local_mean_iwp_lwp = ndimage.convolve(iwp * lwp, footprint, mode='mirror')
local_mean_iwp_w   = ndimage.convolve(iwp * max_w, footprint, mode='mirror')
local_mean_lwp_w   = ndimage.convolve(lwp * max_w, footprint, mode='mirror')

local_cov_iwp_lwp = local_mean_iwp_lwp - local_mean_iwp * local_mean_lwp
local_cov_iwp_w   = local_mean_iwp_w   - local_mean_iwp * local_mean_w
local_cov_lwp_w   = local_mean_lwp_w   - local_mean_lwp * local_mean_w

metric_C = local_cov_iwp_lwp + local_cov_iwp_w + local_cov_lwp_w
# optionally abs: metric_C_abs = np.abs(metric_C)

# --- Choose which metric to use ---
# for example, use metric_A_abs to find point where all three are jointly extreme:
#metric_to_use = metric_A_abs   # replace with metric_B or metric_C as desired
metric_to_use = metric_A   # replace with metric_B or metric_C as desired

# find grid point of maximum metric
flat_index = np.nanargmax(metric_to_use)
x_idx_cov, y_idx_cov = np.unravel_index(flat_index, metric_to_use.shape)

print("chosen indices (x_idx, y_idx):", x_idx_cov, y_idx_cov)
print("x,y coords (km):", x[x_idx_cov]*1e-3, y[y_idx_cov]*1e-3)

x_idx = x_idx_cov
y_idx = y_idx_cov

# Find the index of the maximum value (flattened)
#max_index = np.argmax(max_w)
#max_index = np.argmax(iwp)
# Convert the flattened index back to 2D coordinates
#x_idx, y_idx = np.unravel_index(max_index, max_w.shape)

mean_temp = np.mean(temp,axis=(0,1))-273.15
zero_val,zero_id = find_nearest(mean_temp,0)
neg5_val,neg5_id = find_nearest(mean_temp,-5)
neg10_val,neg10_id = find_nearest(mean_temp,-10)
neg15_val,neg15_id = find_nearest(mean_temp,-15)

In [ ]:
fig = plt.figure(figsize=(20,9),constrained_layout=True)
Fontsize=11
pad=0.05
ax1 = fig.add_subplot(351)
ax2 = fig.add_subplot(352)
ax3 = fig.add_subplot(353)
ax4 = fig.add_subplot(354)
ax5 = fig.add_subplot(355)
ax6 = fig.add_subplot(356)
ax7 = fig.add_subplot(357)
ax8 = fig.add_subplot(358)
ax9 = fig.add_subplot(359)
ax10 = fig.add_subplot(3,5,10)
ax11 = fig.add_subplot(3,5,11)
ax12 = fig.add_subplot(3,5,12)
ax13 = fig.add_subplot(3,5,13)
ax14 = fig.add_subplot(3,5,14)
ax15 = fig.add_subplot(3,5,15)
axlist = [ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10,ax11,ax12,ax13,ax14,ax15]
for ax in axlist:
    ax.grid(which='both',lw=1,ls='dotted',c='dimgrey')
    ax.tick_params(labelsize=Fontsize)

for ax in axlist[0:5]:
    ax.set_xlabel('X [km]',fontsize=Fontsize)
    ax.set_ylabel('Y [km]',fontsize=Fontsize)
for ax in axlist[5:]:
    ax.set_ylim(0,9)
    ax.set_ylabel('Height [km]',fontsize=Fontsize)
    ax.set_xlabel('X [km]',fontsize=Fontsize)

for ax in axlist[0:5]:
    ax.axhline(y[y_idx]*1.e-3, c='magenta', lw=2, ls='dashed',label='Max. Cov(IWP,LWP,$w$)')
    ax.axvline(x[x_idx]*1.e-3, c='magenta', lw=2, ls='dashed')

for ax in axlist[5:]:
    ax.axhline(z[zero_id]*1.e-3,c='maroon',lw=2,ls='dashed',label='0$^{\\circ}$C')
    ax.axhline(z[neg5_id]*1.e-3,c='red',lw=2,ls='dashed',label='-5$^{\\circ}$C')
    ax.axhline(z[neg10_id]*1.e-3,c='darkorange',lw=2,ls='dashed',label='-10$^{\\circ}$C')
    ax.axhline(z[neg15_id]*1.e-3,c='blue',lw=2,ls='dashed',label='-15$^{\\circ}$C')

ax6.legend(loc='lower right',fontsize=Fontsize,labelspacing=0.01)

#============================================
# Horizontal Plots
#============================================
iwp_plot=ax1.contourf(x*1.e-3,y*1.e-3,iwp.T,cmap='Blues',levels=10.**np.arange(-4,2.1,0.1),extend='both',norm=mpl.colors.LogNorm())
cbar=plt.colorbar(iwp_plot,pad=pad,ticks=10.**np.arange(-4,3,1))
cbar.ax.set_ylabel('IWP [g m$^{-2}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

lwp_plot=ax2.contourf(x*1.e-3,y*1.e-3,lwp.T,cmap='Reds',levels=10.**np.arange(0,4.1,0.1),extend='both',norm=mpl.colors.LogNorm())
cbar=plt.colorbar(lwp_plot,pad=pad,ticks=10.**np.arange(0,5,1))
cbar.ax.set_ylabel('LWP [g m$^{-2}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

max_w_plot=ax3.contourf(x*1.e-3,y*1.e-3,max_w.T,cmap='bone_r',levels=np.arange(0,15.5,0.5),extend='max')
cbar=plt.colorbar(max_w_plot,pad=pad,ticks=np.arange(0,20,5))
cbar.ax.set_ylabel('Col.-Max. $w$ [m s$^{-1}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

max_ni_plot=ax4.contourf(x*1.e-3,y*1.e-3,max_ni.T*1.e3,cmap='nipy_spectral',levels=10.**np.arange(-3,2.1,0.1),norm=mpl.colors.LogNorm(),extend='both')
cbar=plt.colorbar(max_ni_plot,pad=pad,ticks=10.**np.arange(-3,3,1))
cbar.ax.set_ylabel('Col.-Max. $N_{ice}$ [L$^{-1}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

max_nl_plot=ax5.contourf(x*1.e-3,y*1.e-3,max_nl.T,cmap='nipy_spectral',levels=10.**np.arange(0,3.1,0.1),norm=mpl.colors.LogNorm(),extend='both')
cbar=plt.colorbar(max_nl_plot,pad=pad,ticks=10.**np.arange(0,4,1))
cbar.ax.set_ylabel('Col.-Max. $N_{liq}$ [cm$^{-3}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)


#============================================
# Vertical Plots
#============================================
#iwc_plot = ax6.contourf(x*1.e-3,z*1.e-3,iwc[:,y_idx,:].T,levels=10.**np.arange(-6,0.1,0.1),norm=mpl.colors.LogNorm(),cmap='Blues',extend='both')
iwc_plot = ax6.contourf(x*1.e-3,z*1.e-3,iwc[:,y_idx,:].T,levels=10.**np.arange(-6,1.1,0.1),norm=mpl.colors.LogNorm(),cmap='nipy_spectral',extend='max')
cbar=plt.colorbar(iwc_plot,pad=pad,ticks=10.**np.arange(-6,2,1))
cbar.ax.set_ylabel('IWC [g m$^{-3}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

lwc_plot = ax7.contourf(x*1.e-3,z*1.e-3,lwc[:,y_idx,:].T,levels=10.**np.arange(-3,1.1,0.1),norm=mpl.colors.LogNorm(),cmap='Reds',extend='both')
cbar=plt.colorbar(lwc_plot,pad=pad,ticks=10.**np.arange(-3,2,1))
cbar.ax.set_ylabel('LWC [g m$^{-3}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

w_plot = ax8.contourf(x*1.e-3,zw*1.e-3,w[:,y_idx,:].T,levels=np.arange(0,15.1,0.1),cmap='bone_r',extend='max')
cbar=plt.colorbar(w_plot,pad=pad,ticks=np.arange(0,20,5))
cbar.ax.set_ylabel('$w$ [m s$^{-1}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

ni_plot = ax9.contourf(x*1.e-3,z*1.e-3,ni[:,y_idx,:].T*1.e3,levels=10.**np.arange(-3,2.1,0.1),norm=mpl.colors.LogNorm(),cmap='nipy_spectral',extend='both')
cbar=plt.colorbar(ni_plot,pad=pad,ticks=10.**np.arange(-3,3,1))
cbar.ax.set_ylabel('$N_{ice}$ [L$^{-1}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

nl_plot = ax10.contourf(x*1.e-3,z*1.e-3,nl[:,y_idx,:].T,levels=10.**np.arange(0,3.1,0.1),norm=mpl.colors.LogNorm(),cmap='nipy_spectral',extend='both')
cbar=plt.colorbar(nl_plot,pad=pad,ticks=10.**np.arange(0,4,1))
cbar.ax.set_ylabel('$N_{liq}$ [cm$^{-3}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

mmdi_plot = ax11.contourf(x*1.e-3,z*1.e-3,mmd_i[:,y_idx,:].T,levels=10.**np.arange(2,4.05,0.05),norm=mpl.colors.LogNorm(),cmap='nipy_spectral',extend='both')
cbar=plt.colorbar(mmdi_plot,pad=pad,ticks=10.**np.arange(2,5,1))
cbar.ax.set_ylabel('MMD$_{ice}$ [$\\mu$m]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

mmdl_plot = ax12.contourf(x*1.e-3,z*1.e-3,mmd_l[:,y_idx,:].T,levels=10.**np.arange(2,4.05,0.05),norm=mpl.colors.LogNorm(),cmap='nipy_spectral',extend='both')
cbar=plt.colorbar(mmdl_plot,pad=pad,ticks=10.**np.arange(2,5,1))
cbar.ax.set_ylabel('MMD$_{liq}$ [$\\mu$m]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

dbz_plot = ax13.contourf(x*1.e-3,z*1.e-3,dBZ_tot[:,y_idx,:].T,levels=np.arange(0,61,1),cmap='nipy_spectral',extend='max')
cbar=plt.colorbar(dbz_plot,pad=pad,ticks=np.arange(0,70,10))
cbar.ax.set_ylabel('Rayleigh Reflectivity [dBZ]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)


reffi_plot = ax14.contourf(x*1.e-3,z*1.e-3,reff_i[:,y_idx,:].T,levels=10.**np.arange(1,4.1,0.1),norm=mpl.colors.LogNorm(),cmap='nipy_spectral',extend='both')
cbar=plt.colorbar(reffi_plot,pad=pad,ticks=10.**np.arange(1,5,1))
cbar.ax.set_ylabel('$R_{eff,ice}$ [$\\mu$m]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

reffl_plot = ax15.contourf(x*1.e-3,z*1.e-3,reff_l[:,y_idx,:].T,levels=10.**np.arange(1,4.1,0.1),norm=mpl.colors.LogNorm(),cmap='nipy_spectral',extend='both')
cbar=plt.colorbar(reffl_plot,pad=pad,ticks=10.**np.arange(1,5,1))
cbar.ax.set_ylabel('$R_{eff,liq}$ [$\\mu$m]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)




labs = ['IWP','LWP','Col-Max $w$','Col-Max $N_{ice}$','Col-Max $N_{liq}$',\
        'IWC','LWC','$w$','$N_{ice}$','$N_{liq}$',\
        'MMD$_{ice}$','MMD$_{liq}$','dBZ','$R_{eff,ice}$','$R_{eff,liq}$']
props = dict(boxstyle='square', facecolor='wheat', alpha=0.8)

ii=0
for ax in axlist:
    ax.text(0.97,0.97,labs[ii],fontsize=Fontsize*1.5,fontweight='bold',c='k',transform=ax.transAxes,bbox=props,ha='right',va='top')
    ii+=1

lab = 'Max. $Cov$(IWP, LWP, Col-Max $w$)'
#lab = 'Max $w$'
legend_line = Line2D([0], [0],color='magenta', linewidth=2,linestyle='dashed')
leg=ax1.legend(handles=[legend_line],labels=[lab],fontsize=Fontsize*1.25,framealpha=False,bbox_to_anchor=(-0.1, 1.2),loc='upper left')
plt.suptitle('Slice @ '+lab+' @ '+str(np.around(var_dict['time']/3600.,3))+' hrs',fontsize=Fontsize*1.5,fontweight='bold')
leg.set_in_layout(False)

plt.show()
plt.close()

In [ ]:
var_dict['time']

In [ ]:
fig = plt.figure(figsize=(20,5.5),constrained_layout=True)
Fontsize=10
pad=0.05
ax1 = fig.add_subplot(251)
ax2 = fig.add_subplot(252)
ax3 = fig.add_subplot(253)
ax4 = fig.add_subplot(254)
ax5 = fig.add_subplot(255)
ax6 = fig.add_subplot(256)
ax7 = fig.add_subplot(257)
ax8 = fig.add_subplot(258)
ax9 = fig.add_subplot(259)
ax10 = fig.add_subplot(2,5,10)
axlist = [ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10]
for ax in axlist:
    ax.grid(which='both',lw=1,ls='dotted',c='dimgrey')
    ax.tick_params(labelsize=Fontsize)

for ax in axlist[0:5]:
    ax.set_xlabel('X [km]',fontsize=Fontsize)
    ax.set_ylabel('Y [km]',fontsize=Fontsize)
for ax in axlist[5:]:
    ax.set_ylim(0,8)
    ax.set_ylabel('Height [km]',fontsize=Fontsize)
    ax.set_xlabel('X [km]',fontsize=Fontsize)

for ax in axlist[0:5]:
    ax.axhline(y[y_idx]*1.e-3, c='magenta', lw=2, ls='dashed')
    ax.axvline(x[x_idx]*1.e-3, c='magenta', lw=2, ls='dashed')

for ax in axlist[5:]:
    ax.axhline(z[zero_id]*1.e-3,c='maroon',lw=2,ls='dashed',label='0$^{\\circ}$C')
    ax.axhline(z[neg5_id]*1.e-3,c='red',lw=2,ls='dashed',label='-5$^{\\circ}$C')
    ax.axhline(z[neg10_id]*1.e-3,c='darkorange',lw=2,ls='dashed',label='-10$^{\\circ}$C')
    ax.axhline(z[neg15_id]*1.e-3,c='blue',lw=2,ls='dashed',label='-15$^{\\circ}$C')

ax6.legend(loc='lower right',fontsize=Fontsize,labelspacing=0.01)

#============================================
# Horizontal Plots
#============================================
iwp_plot=ax1.contourf(x*1.e-3,y*1.e-3,iwp.T,cmap='Blues',levels=10.**np.arange(-4,1.1,0.1),extend='both',norm=mpl.colors.LogNorm())
cbar=plt.colorbar(iwp_plot,pad=pad,ticks=10.**np.arange(-4,2,1))
cbar.ax.set_ylabel('IWP [g m$^{-2}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

lwp_plot=ax2.contourf(x*1.e-3,y*1.e-3,lwp.T,cmap='Reds',levels=10.**np.arange(0,4.1,0.1),extend='both',norm=mpl.colors.LogNorm())
cbar=plt.colorbar(lwp_plot,pad=pad,ticks=10.**np.arange(0,5,1))
cbar.ax.set_ylabel('LWP [g m$^{-2}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

max_w_plot=ax3.contourf(x*1.e-3,y*1.e-3,max_w.T,cmap='bone_r',levels=np.arange(0,15.5,0.5),extend='max')
cbar=plt.colorbar(max_w_plot,pad=pad,ticks=np.arange(0,20,5))
cbar.ax.set_ylabel('Col.-Max. $w$ [m s$^{-1}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

max_ni_plot=ax4.contourf(x*1.e-3,y*1.e-3,max_ni.T*1.e3,cmap='nipy_spectral',levels=10.**np.arange(-3,0.1,0.1),norm=mpl.colors.LogNorm(),extend='both')
cbar=plt.colorbar(max_ni_plot,pad=pad,ticks=10.**np.arange(-3,1,1))
cbar.ax.set_ylabel('Col.-Max. $N_{ice}$ [L$^{-1}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

max_nl_plot=ax5.contourf(x*1.e-3,y*1.e-3,max_nl.T,cmap='nipy_spectral',levels=10.**np.arange(0,3.1,0.1),norm=mpl.colors.LogNorm(),extend='both')
cbar=plt.colorbar(max_nl_plot,pad=pad,ticks=10.**np.arange(0,4,1))
cbar.ax.set_ylabel('Col.-Max. $N_{liq}$ [cm$^{-3}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)


#============================================
# Vertical Plots
#============================================
iwc_plot = ax6.contourf(x*1.e-3,z*1.e-3,iwc[:,y_idx,:].T,levels=10.**np.arange(-6,-1.9,0.1),norm=mpl.colors.LogNorm(),cmap='Blues',extend='both')
cbar=plt.colorbar(iwc_plot,pad=pad,ticks=10.**np.arange(-6,-1,1))
cbar.ax.set_ylabel('IWC [g m$^{-3}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

lwc_plot = ax7.contourf(x*1.e-3,z*1.e-3,lwc[:,y_idx,:].T,levels=10.**np.arange(-3,1.1,0.1),norm=mpl.colors.LogNorm(),cmap='Reds',extend='both')
cbar=plt.colorbar(lwc_plot,pad=pad,ticks=10.**np.arange(-3,2,1))
cbar.ax.set_ylabel('LWC [g m$^{-3}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

w_plot = ax8.contourf(x*1.e-3,zw*1.e-3,w[:,y_idx,:].T,levels=np.arange(0,15.1,0.1),cmap='bone_r',extend='max')
cbar=plt.colorbar(w_plot,pad=pad,ticks=np.arange(0,20,5))
cbar.ax.set_ylabel('$w$ [m s$^{-1}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

ni_plot = ax9.contourf(x*1.e-3,z*1.e-3,ni[:,y_idx,:].T*1.e3,levels=10.**np.arange(-3,0.1,0.1),norm=mpl.colors.LogNorm(),cmap='nipy_spectral',extend='both')
cbar=plt.colorbar(ni_plot,pad=pad,ticks=10.**np.arange(-3,1,1))
cbar.ax.set_ylabel('$N_{ice}$ [L$^{-1}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

nl_plot = ax10.contourf(x*1.e-3,z*1.e-3,nl[:,y_idx,:].T,levels=10.**np.arange(0,3.1,0.1),norm=mpl.colors.LogNorm(),cmap='nipy_spectral',extend='both')
cbar=plt.colorbar(nl_plot,pad=pad,ticks=10.**np.arange(0,4,1))
cbar.ax.set_ylabel('$N_{liq}$ [cm$^{-3}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

plt.suptitle('Slice @ Max Covariance of IWP, LWP, & Col.-Max. $w$',fontsize=Fontsize*1.5,fontweight='bold')
#plt.suptitle('Slice @ Max $w$',fontsize=Fontsize*1.5,fontweight='bold')
#plt.suptitle('Slice @ Max IWP',fontsize=Fontsize*1.5,fontweight='bold')

plt.show()
plt.close()

In [ ]:
fig = plt.figure(figsize=(20,5.5),constrained_layout=True)
Fontsize=10
pad=0.05
ax1 = fig.add_subplot(251)
ax2 = fig.add_subplot(252)
ax3 = fig.add_subplot(253)
ax4 = fig.add_subplot(254)
ax5 = fig.add_subplot(255)
ax6 = fig.add_subplot(256)
ax7 = fig.add_subplot(257)
ax8 = fig.add_subplot(258)
ax9 = fig.add_subplot(259)
ax10 = fig.add_subplot(2,5,10)
axlist = [ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10]
for ax in axlist:
    ax.grid(which='both',lw=1,ls='dotted',c='dimgrey')
    ax.tick_params(labelsize=Fontsize)

for ax in axlist[0:5]:
    ax.set_xlabel('X [km]',fontsize=Fontsize)
    ax.set_ylabel('Y [km]',fontsize=Fontsize)
for ax in axlist[5:]:
    ax.set_ylim(0,8)
    ax.set_ylabel('Height [km]',fontsize=Fontsize)
    ax.set_xlabel('X [km]',fontsize=Fontsize)

for ax in axlist[0:5]:
    ax.axhline(y[y_idx]*1.e-3, c='magenta', lw=2, ls='dashed')
    ax.axvline(x[x_idx]*1.e-3, c='magenta', lw=2, ls='dashed')

for ax in axlist[5:]:
    ax.axhline(z[zero_id]*1.e-3,c='maroon',lw=2,ls='dashed',label='0$^{\\circ}$C')
    ax.axhline(z[neg5_id]*1.e-3,c='red',lw=2,ls='dashed',label='-5$^{\\circ}$C')
    ax.axhline(z[neg10_id]*1.e-3,c='darkorange',lw=2,ls='dashed',label='-10$^{\\circ}$C')
    ax.axhline(z[neg15_id]*1.e-3,c='blue',lw=2,ls='dashed',label='-15$^{\\circ}$C')

ax6.legend(loc='lower right',fontsize=Fontsize,labelspacing=0.01)

#============================================
# Horizontal Plots
#============================================
iwp_plot=ax1.contourf(x*1.e-3,y*1.e-3,iwp.T,cmap='nipy_spectral',levels=10.**np.arange(-4,2.1,0.1),extend='both',norm=mpl.colors.LogNorm())
cbar=plt.colorbar(iwp_plot,pad=pad,ticks=10.**np.arange(-4,3,1))
cbar.ax.set_ylabel('IWP [g m$^{-2}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

lwp_plot=ax2.contourf(x*1.e-3,y*1.e-3,lwp.T,cmap='nipy_spectral',levels=10.**np.arange(0,4.1,0.1),extend='both',norm=mpl.colors.LogNorm())
cbar=plt.colorbar(lwp_plot,pad=pad,ticks=10.**np.arange(0,5,1))
cbar.ax.set_ylabel('LWP [g m$^{-2}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

max_w_plot=ax3.contourf(x*1.e-3,y*1.e-3,max_w.T,cmap='nipy_spectral',levels=np.arange(2,20.5,0.5),extend='max')
cbar=plt.colorbar(max_w_plot,pad=pad,ticks=np.arange(2,22,2))
cbar.ax.set_ylabel('Col.-Max. $w$ [m s$^{-1}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

max_ni_plot=ax4.contourf(x*1.e-3,y*1.e-3,max_ni.T*1.e3,cmap='nipy_spectral',levels=10.**np.arange(-3,0.1,0.1),norm=mpl.colors.LogNorm(),extend='both')
cbar=plt.colorbar(max_ni_plot,pad=pad,ticks=10.**np.arange(-3,1,1))
cbar.ax.set_ylabel('Col.-Max. $N_{ice}$ [L$^{-1}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

max_nl_plot=ax5.contourf(x*1.e-3,y*1.e-3,max_nl.T,cmap='nipy_spectral',levels=10.**np.arange(0,3.1,0.1),norm=mpl.colors.LogNorm(),extend='both')
cbar=plt.colorbar(max_nl_plot,pad=pad,ticks=10.**np.arange(0,4,1))
cbar.ax.set_ylabel('Col.-Max. $N_{liq}$ [cm$^{-3}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)


#============================================
# Vertical Plots
#============================================
iwc_plot = ax6.contourf(x*1.e-3,z*1.e-3,iwc[:,y_idx,:].T,levels=10.**np.arange(-6,-0.9,0.1),norm=mpl.colors.LogNorm(),cmap='nipy_spectral',extend='both')
cbar=plt.colorbar(iwc_plot,pad=pad,ticks=10.**np.arange(-6,0,1))
cbar.ax.set_ylabel('IWC [g m$^{-3}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

lwc_plot = ax7.contourf(x*1.e-3,z*1.e-3,lwc[:,y_idx,:].T,levels=10.**np.arange(-3,1.1,0.1),norm=mpl.colors.LogNorm(),cmap='nipy_spectral',extend='both')
cbar=plt.colorbar(lwc_plot,pad=pad,ticks=10.**np.arange(-3,2,1))
cbar.ax.set_ylabel('LWC [g m$^{-3}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

w_plot = ax8.contourf(x*1.e-3,zw*1.e-3,w[:,y_idx,:].T,levels=np.arange(2,20.1,0.1),cmap='nipy_spectral',extend='max')
cbar=plt.colorbar(w_plot,pad=pad,ticks=np.arange(2,22,2))
cbar.ax.set_ylabel('$w$ [m s$^{-1}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

ni_plot = ax9.contourf(x*1.e-3,z*1.e-3,ni[:,y_idx,:].T*1.e3,levels=10.**np.arange(-3,0.1,0.1),norm=mpl.colors.LogNorm(),cmap='nipy_spectral',extend='both')
cbar=plt.colorbar(ni_plot,pad=pad,ticks=10.**np.arange(-3,1,1))
cbar.ax.set_ylabel('$N_{ice}$ [L$^{-1}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

nl_plot = ax10.contourf(x*1.e-3,z*1.e-3,nl[:,y_idx,:].T,levels=10.**np.arange(0,3.1,0.1),norm=mpl.colors.LogNorm(),cmap='nipy_spectral',extend='both')
cbar=plt.colorbar(nl_plot,pad=pad,ticks=10.**np.arange(0,4,1))
cbar.ax.set_ylabel('$N_{liq}$ [cm$^{-3}$]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

plt.suptitle('Slice @ Max Covariance of IWP, LWP, & Col.-Max. $w$',fontsize=Fontsize*1.5,fontweight='bold')
#plt.suptitle('Slice @ Max $w$',fontsize=Fontsize*1.5,fontweight='bold')
#plt.suptitle('Slice @ Max IWP',fontsize=Fontsize*1.5,fontweight='bold')

plt.show()
plt.close()

In [ ]:
np.max(w)